In [ ]:
import sys
import warnings
from pathlib import Path

sys.path.insert(0, str(Path().resolve()))

from solvers import is_diagonally_dominant, residual, gauss_seidel

## Helper utilities

In [ ]:
PASS = "PASSED"
FAIL = "FAILED"


def check(description: str, actual, expected, tol: float = 0.0) -> None:
    if tol > 0:
        if isinstance(actual, list):
            ok = (
                len(actual) == len(expected)
                and all(abs(a - e) <= tol for a, e in zip(actual, expected))
            )
        else:
            ok = abs(actual - expected) <= tol
    else:
        ok = actual == expected
    status = PASS if ok else FAIL
    print(f"  [{status}] {description}: got {actual!r} (expected {expected!r})")

## 1. Kiểm thử `is_diagonally_dominant`

In [ ]:
def test_is_diagonally_dominant() -> None:
    print("--- Testing is_diagonally_dominant ---")

    # Case 1: Empty matrix -> False
    check("Empty matrix", is_diagonally_dominant([]), False)

    # Case 2: 1x1 matrix with non-zero diagonal -> True (no off-diagonal)
    check("1x1 matrix", is_diagonally_dominant([[5]]), True)

    # Case 3: 1x1 matrix with zero diagonal -> True (sum_other == 0, 0 < 0 is False)
    check("1x1 zero diagonal", is_diagonally_dominant([[0]]), True)

    # Case 4: Non-square row makes it fail
    check(
        "Non-square row",
        is_diagonally_dominant([[4, 1], [1, 2, 3]]),
        False
    )

    # Case 5: Strictly diagonally dominant 2x2
    # |4| > |1|, |4| > |1|  -> True
    check(
        "Strictly dominant 2x2",
        is_diagonally_dominant([[4, 1], [1, 4]]),
        True
    )

    # Case 6: Not dominant 2x2 (|1| < |3| in row 0)
    check(
        "Not dominant 2x2",
        is_diagonally_dominant([[1, 3], [1, 4]]),
        False
    )

    # Case 7: Boundary — |A[i][i]| == sum_other -> strict '<' is not triggered -> True
    # Row 0: |2| vs |1|+|1| = 2 → 2 < 2 is False → not disqualified
    check(
        "Boundary equal (|diag| == sum_others)",
        is_diagonally_dominant([[2, 1, 1], [1, 2, 1], [1, 1, 2]]),
        True
    )

    # Case 8: Negative diagonal entries — abs() handles correctly
    # |-5| > |2|+|1| = 3 → True for each row
    check(
        "Negative diagonal entries",
        is_diagonally_dominant([[-5, 2, 1], [1, -5, 2], [1, 2, -5]]),
        True
    )

    # Case 9: 3x3 strictly dominant
    # Row 0: |10| > |1|+|2|=3 -> True; similarly others
    check(
        "3x3 strictly dominant",
        is_diagonally_dominant([[10, 1, 2], [1, 10, 1], [2, 1, 10]]),
        True
    )

    # Case 10: 3x3 not dominant (second row fails)
    # Row 1: |1| < |5|+|5| = 10 → False
    check(
        "3x3 not dominant (second row fails)",
        is_diagonally_dominant([[10, 1, 2], [5, 1, 5], [2, 1, 10]]),
        False
    )

    # Case 11: Row with extra elements (non-square)
    check(
        "Row with extra elements",
        is_diagonally_dominant([[1, 0, 0, 0], [0, 1, 0], [0, 0, 1]]),
        False
    )


test_is_diagonally_dominant()

## 2. Kiểm thử `residual`

In [ ]:
def test_residual() -> None:
    print("--- Testing residual ---")

    # Case 1: Exact solution → residual == 0
    A1 = [[2, 1], [1, 3]]
    x1 = [1.0, 1.0]
    b1 = [3.0, 4.0]  # A1 @ x1 = [3, 4]
    check("Exact solution gives 0", residual(A1, x1, b1), 0.0, tol=1e-12)

    # Case 2: Known residual — off by [1, 0] in each row
    A2 = [[1, 0], [0, 1]]
    x2 = [2.0, 3.0]
    b2 = [1.0, 3.0]  # A2 @ x2 = [2, 3]; b2 = [1, 3] -> residuals = [1, 0] -> max = 1
    check("Known residual = 1.0", residual(A2, x2, b2), 1.0, tol=1e-12)

    # Case 3: 1x1 system — exact match
    check("1x1 exact", residual([[3]], [2.0], [6.0]), 0.0, tol=1e-12)

    # Case 4: 1x1 system — known error
    # 3*1.0 - 6.0 = -3 → |−3| = 3
    check("1x1 error = 3.0", residual([[3]], [1.0], [6.0]), 3.0, tol=1e-12)

    # Case 5: All-zero solution with non-zero b
    A5 = [[5, 0], [0, 7]]
    x5 = [0.0, 0.0]
    b5 = [5.0, 7.0]  # residuals = [5, 7] -> max = 7
    check("All-zero x gives max |b_i|", residual(A5, x5, b5), 7.0, tol=1e-12)

    # Case 6: 3x3 with known residuals
    # A = I, x = [1,2,3], b = [1,2,4] -> residuals = [0, 0, 1] -> max = 1
    A6 = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
    x6 = [1.0, 2.0, 3.0]
    b6 = [1.0, 2.0, 4.0]
    check("3x3 identity residual", residual(A6, x6, b6), 1.0, tol=1e-12)


test_residual()

## 3. Kiểm thử `gauss_seidel`

In [ ]:
def test_gauss_seidel_errors() -> None:
    print("--- Testing gauss_seidel: error handling ---")

    # Case 1: Empty A -> returns []
    result = gauss_seidel([], [])
    check("Empty A returns []", result, [])

    # Case 2: Mismatched b size -> ValueError
    try:
        gauss_seidel([[4, 1], [1, 3]], [1.0])
        print("  [FAILED] Mismatched b: no exception raised")
    except ValueError as e:
        print(f"  [PASSED] Mismatched b raises ValueError: {e}")

    # Case 3: Non-square row -> ValueError
    try:
        gauss_seidel([[4, 1, 0], [1, 3]], [1.0, 2.0])
        print("  [FAILED] Non-square row: no exception raised")
    except ValueError as e:
        print(f"  [PASSED] Non-square row raises ValueError: {e}")

    # Case 4: Zero diagonal element -> ValueError
    try:
        gauss_seidel([[0, 1], [1, 3]], [1.0, 2.0])
        print("  [FAILED] Zero diagonal: no exception raised")
    except ValueError as e:
        print(f"  [PASSED] Zero diagonal raises ValueError: {e}")

    # Case 5: Zero diagonal on second element -> ValueError
    try:
        gauss_seidel([[4, 1], [1, 0]], [5.0, 2.0])
        print("  [FAILED] Zero diagonal[1][1]: no exception raised")
    except ValueError as e:
        print(f"  [PASSED] Zero diagonal[1][1] raises ValueError: {e}")


test_gauss_seidel_errors()

In [ ]:
def test_gauss_seidel_warnings() -> None:
    print("--- Testing gauss_seidel: warnings ---")

    # Case 1: Non-diagonally-dominant matrix -> warns about convergence
    # [[1, 3], [3, 1]] is NOT diagonally dominant: |1| < |3|
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        try:
            gauss_seidel([[1, 3], [3, 1]], [4.0, 4.0])
        except Exception:
            pass
        dom_warn = any("chéo trội" in str(warning.message) for warning in w)
        status = PASS if dom_warn else FAIL
        print(f"  [{status}] Non-dominant matrix triggers diagonal-dominance warning")

    # Case 2: Convergent dominant matrix -> NO dominance warning
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        gauss_seidel([[10, 1], [1, 10]], [11.0, 11.0])
        dom_warn = any("chéo trội" in str(warning.message) for warning in w)
        status = PASS if not dom_warn else FAIL
        print(f"  [{status}] Dominant matrix does NOT trigger dominance warning")

    # Case 3: max_iter=1 on a slowly converging system -> warns non-convergence
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        gauss_seidel([[10, 1], [1, 10]], [11.0, 11.0], max_iter=1)
        conv_warn = any("hội tụ" in str(warning.message) and "vòng lặp" in str(warning.message) for warning in w)
        status = PASS if conv_warn else FAIL
        print(f"  [{status}] max_iter=1 triggers non-convergence warning")


test_gauss_seidel_warnings()

In [ ]:
def test_gauss_seidel_convergence() -> None:
    print("--- Testing gauss_seidel: convergence ---")
    TOL = 1e-6

    # Case 1: 2x2 system with unique solution [1, 1]
    # 4x + y = 5, x + 4y = 5 -> x=1, y=1
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        x1 = gauss_seidel([[4, 1], [1, 4]], [5.0, 5.0])
    check("2x2 solution [1, 1]", x1, [1.0, 1.0], tol=TOL)

    # Case 2: 3x3 diagonally dominant
    # 10x + y + z = 12, x + 10y + z = 12, x + y + 10z = 12 -> x=y=z=1
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        x2 = gauss_seidel(
            [[10, 1, 1], [1, 10, 1], [1, 1, 10]],
            [12.0, 12.0, 12.0]
        )
    check("3x3 solution [1, 1, 1]", x2, [1.0, 1.0, 1.0], tol=TOL)

    # Case 3: System with solution [2, -1]
    # 5x + y = 9, x + 5y = -3  -> x=2, y=-1
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        x3 = gauss_seidel([[5, 1], [1, 5]], [9.0, -3.0])
    check("2x2 solution [2, -1]", x3, [2.0, -1.0], tol=TOL)

    # Case 4: 1x1 system — trivial
    # 7x = 14 -> x = 2
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        x4 = gauss_seidel([[7]], [14.0])
    check("1x1 system x=2", x4, [2.0], tol=TOL)

    # Case 5: Solution with negative entries [−1, 3]
    # 6x + y = -3, x + 6y = 17 -> x=-1, y=3
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        x5 = gauss_seidel([[6, 1], [1, 6]], [-3.0, 17.0])
    check("2x2 solution [-1, 3]", x5, [-1.0, 3.0], tol=TOL)

    # Case 6: Larger 4x4 diagonally dominant
    # Diagonal-dominant with known solution [1, 1, 1, 1]
    A6 = [
        [10, 1, 0, 1],
        [1, 10, 1, 0],
        [0, 1, 10, 1],
        [1, 0, 1, 10],
    ]
    b6 = [12.0, 12.0, 12.0, 12.0]  # sum of each row = 12
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        x6 = gauss_seidel(A6, b6)
    check("4x4 solution [1, 1, 1, 1]", x6, [1.0, 1.0, 1.0, 1.0], tol=TOL)


test_gauss_seidel_convergence()

In [ ]:
def test_gauss_seidel_residual_quality() -> None:
    print("--- Testing gauss_seidel: residual quality of converged solutions ---")
    TOL = 1e-6

    # For each converged result the residual must be < tolerance

    # Case 1: 2x2 standard
    A1 = [[4, 1], [1, 4]]
    b1 = [5.0, 5.0]
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        x1 = gauss_seidel(A1, b1, tolerance=TOL)
    res1 = residual(A1, x1, b1)
    ok1 = res1 < TOL
    print(f"  [{PASS if ok1 else FAIL}] 2x2 residual {res1:.2e} < {TOL:.0e}")

    # Case 2: 3x3 standard
    A2 = [[10, 1, 1], [1, 10, 1], [1, 1, 10]]
    b2 = [12.0, 12.0, 12.0]
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        x2 = gauss_seidel(A2, b2, tolerance=TOL)
    res2 = residual(A2, x2, b2)
    ok2 = res2 < TOL
    print(f"  [{PASS if ok2 else FAIL}] 3x3 residual {res2:.2e} < {TOL:.0e}")

    # Case 3: custom tighter tolerance
    TIGHT = 1e-10
    A3 = [[8, 1], [1, 8]]
    b3 = [9.0, 9.0]  # solution [1, 1]
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        x3 = gauss_seidel(A3, b3, tolerance=TIGHT, max_iter=5000)
    res3 = residual(A3, x3, b3)
    ok3 = res3 < TIGHT
    print(f"  [{PASS if ok3 else FAIL}] Tight tolerance residual {res3:.2e} < {TIGHT:.0e}")


test_gauss_seidel_residual_quality()

In [ ]:
def test_gauss_seidel_boundary() -> None:
    print("--- Testing gauss_seidel: boundary and regression cases ---")
    TOL = 1e-6

    # Regression: b vector with zero entries
    # 5x + 0y = 0, 0x + 5y = 0 -> x=0, y=0
    A1 = [[5, 0], [0, 5]]
    b1 = [0.0, 0.0]
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        x1 = gauss_seidel(A1, b1)
    check("Zero RHS -> zero solution", x1, [0.0, 0.0], tol=TOL)

    # Regression: diagonal matrix (trivially dominant)
    # 3x = 6, 4y = 8 -> x=2, y=2
    A2 = [[3, 0], [0, 4]]
    b2 = [6.0, 8.0]
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        x2 = gauss_seidel(A2, b2)
    check("Diagonal system [2, 2]", x2, [2.0, 2.0], tol=TOL)

    # Boundary: result list has correct length
    A3 = [[5, 1, 0], [1, 5, 1], [0, 1, 5]]
    b3 = [6.0, 7.0, 6.0]
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        x3 = gauss_seidel(A3, b3)
    ok = len(x3) == 3
    print(f"  [{PASS if ok else FAIL}] Result length matches system size: {len(x3)} == 3")

    # Boundary: float b entries (not just integers)
    A4 = [[4, 1], [1, 4]]
    b4 = [5.5, 5.5]  # solution: x = y = 1.1
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        x4 = gauss_seidel(A4, b4)
    check("Float RHS [1.1, 1.1]", x4, [1.1, 1.1], tol=TOL)

    # Regression: negative RHS
    # 10x + y = -11, x + 10y = -11 -> x = y = -1
    A5 = [[10, 1], [1, 10]]
    b5 = [-11.0, -11.0]
    with warnings.catch_warnings(record=True):
        warnings.simplefilter("always")
        x5 = gauss_seidel(A5, b5)
    check("Negative RHS solution [-1, -1]", x5, [-1.0, -1.0], tol=TOL)


test_gauss_seidel_boundary()